# Heart Disease Classification
**Team:** Karim · Kushul · Tarik · Max  
**Institution:** Hochschule Pforzheim — BIS3012  
**Supervisor:** Prof. Dr. Dustin van der Haar  

---

## Project Overview

Heart disease is the world's leading cause of death. This project builds and compares three
machine learning classifiers that predict whether a patient has heart disease based on
17 lifestyle and health measurements collected in the CDC's national health survey.

| Item | Detail |
|---|---|
| **Task** | Binary Classification (0 = No Disease, 1 = Disease) |
| **Dataset** | CDC BRFSS 2020 — 2,059 patients at real-world disease rate (8.6%) |
| **Source** | Centers for Disease Control and Prevention, United States |
| **Models** | Logistic Regression · Random Forest · SVM (RBF) |
| **Tuning** | GridSearchCV — 5-fold stratified cross-validation |
| **Imbalance** | Handled via `class_weight='balanced'` |

---
*Kamilpytlak (2021). Personal Key Indicators of Heart Disease. Kaggle. Derived from CDC BRFSS 2020.*


---
## 1. Setup & Data Loading

### Running in Google Colab
Upload `heart_2020_cleaned.csv` when the first code cell prompts you.  
**Alternative:** If the file is in Google Drive, replace the upload code with:
```python
from google.colab import drive
drive.mount('/content/drive')
df_full = pd.read_csv('/content/drive/MyDrive/heart_2020_cleaned.csv')
```


In [ ]:
# ── Upload the CSV file (Google Colab) ───────────────────────────────────────
from google.colab import files
import io

uploaded = files.upload()          # click Choose Files → select heart_2020_cleaned.csv
filename  = list(uploaded.keys())[0]
print(f"Uploaded: {filename}")


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection  import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing    import StandardScaler
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier
from sklearn.svm              import SVC
from sklearn.metrics          import (accuracy_score, classification_report,
                                       confusion_matrix, roc_curve, auc,
                                       ConfusionMatrixDisplay, f1_score,
                                       precision_score, recall_score)

print("All libraries loaded.")


### Load and Sample the Dataset

The full CDC dataset has **319,795 responses** but is heavily imbalanced: only 8.6% of
respondents report heart disease. We take **2,059 patients** that preserve this exact
real-world ratio — 177 with disease (8.6%) and 1,882 without (91.4%).

**Why keep the imbalance?**  
A 50/50 split would misrepresent reality and would not reflect how the model performs
in the real world. Instead we keep the natural distribution and handle the imbalance
correctly using `class_weight='balanced'` when training each model.


In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
df_full = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"Full dataset:  {df_full.shape[0]:,} patients, {df_full.shape[1]} features")
print(f"Disease rate:  {(df_full['HeartDisease']=='Yes').mean()*100:.1f}%")

# ── Realistic sample of 2,059 patients ───────────────────────────────────────
# Preserves the real CDC disease rate (~8.6%)
disease    = df_full[df_full['HeartDisease'] == 'Yes'].sample(177,  random_state=42)
no_disease = df_full[df_full['HeartDisease'] == 'No'].sample(1882, random_state=42)

df = (pd.concat([disease, no_disease])
        .sample(frac=1, random_state=42)
        .reset_index(drop=True))

print(f"\nSample:        {len(df)} patients")
print(f"  Disease:     {(df['HeartDisease']=='Yes').sum()} ({(df['HeartDisease']=='Yes').mean()*100:.1f}%)")
print(f"  No Disease:  {(df['HeartDisease']=='No').sum()} ({(df['HeartDisease']=='No').mean()*100:.1f}%)")
print(f"  Missing:     {df.isnull().sum().sum()}")
df.head()


---
## 2. Exploratory Data Analysis (EDA)

We explore the data before training anything. The goal is to understand distributions,
identify patterns, and spot any issues.


### 2.1 Dataset Overview

In [ ]:
# Data types and non-null counts
print("=== Column Types ===")
df.info()
print()
df.describe(include='all').round(2)


### 2.2 Missing Values

All values should be zero before we proceed.


In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "None — dataset is complete.")
print(f"\nTotal: {missing.sum()}")


### 2.3 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['HeartDisease'].value_counts()
no_count  = counts.get('No', 0)
yes_count = counts.get('Yes', 0)

# Bar chart
axes[0].bar(['No Disease', 'Heart Disease'], [no_count, yes_count],
            color=['#15803d', '#b91c1c'], edgecolor='black', width=0.5)
axes[0].set_title('Patient Count by Class')
axes[0].set_ylabel('Patients')
for i, v in enumerate([no_count, yes_count]):
    axes[0].text(i, v + 15, str(v), ha='center', fontweight='bold')
axes[0].set_ylim(0, no_count * 1.15)

# Pie
axes[1].pie([no_count, yes_count],
            labels=[f'No Disease
({no_count})', f'Heart Disease
({yes_count})'],
            autopct='%1.1f%%', colors=['#15803d', '#b91c1c'],
            startangle=90, pctdistance=0.75)
axes[1].set_title('Real-World CDC Disease Rate')

plt.suptitle(f'CDC BRFSS 2020 Sample — {len(df)} Patients', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Note: 8.6% disease rate reflects the real CDC population distribution.")
print("Imbalance is handled via class_weight='balanced' during model training.")


### 2.4 Disease Rate by Age Group

Age is expected to be the strongest predictor — disease rates should increase with age.


In [ ]:
age_order = ['18-24','25-29','30-34','35-39','40-44','45-49',
             '50-54','55-59','60-64','65-69','70-74','75-79','80 or older']

# Disease rate per age group
age_rate = (df.groupby('AgeCategory')['HeartDisease']
              .apply(lambda x: (x=='Yes').mean() * 100)
              .reindex(age_order)
              .dropna())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Disease rate line
axes[0].plot(range(len(age_rate)), age_rate.values, marker='o',
             color='#b91c1c', lw=2, ms=7)
axes[0].fill_between(range(len(age_rate)), age_rate.values, alpha=0.15, color='#b91c1c')
axes[0].set_xticks(range(len(age_rate)))
axes[0].set_xticklabels(age_rate.index, rotation=45, ha='right')
axes[0].set_ylabel('Disease Rate (%)')
axes[0].set_title('Heart Disease Rate by Age Group')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(age_rate.values):
    axes[0].annotate(f'{v:.1f}%', (i, v), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=8)

# Count bars
age_counts = df.groupby(['AgeCategory','HeartDisease']).size().unstack(fill_value=0)
age_counts = age_counts.reindex([a for a in age_order if a in age_counts.index])
age_counts.plot(kind='bar', ax=axes[1], color=['#15803d','#b91c1c'],
                edgecolor='black', width=0.75)
axes[1].set_title('Patient Count by Age Group')
axes[1].set_xlabel('')
axes[1].legend(['No Disease','Heart Disease'])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()


### 2.5 Key Risk Factors

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# BMI
for label, color, lbl in [('No','#15803d','No Disease'),('Yes','#b91c1c','Heart Disease')]:
    axes[0,0].hist(df[df['HeartDisease']==label]['BMI'],
                   bins=30, alpha=0.6, color=color, label=lbl, edgecolor='white')
axes[0,0].set_title('BMI Distribution')
axes[0,0].set_xlabel('BMI')
axes[0,0].legend()

# General Health
gh_order = ['Poor','Fair','Good','Very good','Excellent']
gh_data = df.groupby(['GenHealth','HeartDisease']).size().unstack(fill_value=0)
gh_data = gh_data.reindex([g for g in gh_order if g in gh_data.index])
gh_data.plot(kind='bar', ax=axes[0,1], color=['#15803d','#b91c1c'],
             edgecolor='black', rot=30)
axes[0,1].set_title('General Health vs Disease')
axes[0,1].legend(['No Disease','Heart Disease'])

# Physical Health days
for label, color, lbl in [('No','#15803d','No Disease'),('Yes','#b91c1c','Heart Disease')]:
    axes[0,2].hist(df[df['HeartDisease']==label]['PhysicalHealth'],
                   bins=20, alpha=0.6, color=color, label=lbl, edgecolor='white')
axes[0,2].set_title('Poor Physical Health Days')
axes[0,2].set_xlabel('Days in last 30')
axes[0,2].legend()

# Smoking
pd.crosstab(df['Smoking'], df['HeartDisease']).plot(
    kind='bar', ax=axes[1,0], color=['#15803d','#b91c1c'],
    edgecolor='black', rot=0)
axes[1,0].set_title('Smoking vs Disease')
axes[1,0].legend(['No Disease','Heart Disease'])

# Diabetes
pd.crosstab(df['Diabetic'], df['HeartDisease']).plot(
    kind='bar', ax=axes[1,1], color=['#15803d','#b91c1c'],
    edgecolor='black', rot=20)
axes[1,1].set_title('Diabetic Status vs Disease')
axes[1,1].legend(['No Disease','Heart Disease'])

# Walking difficulty
pd.crosstab(df['DiffWalking'], df['HeartDisease']).plot(
    kind='bar', ax=axes[1,2], color=['#15803d','#b91c1c'],
    edgecolor='black', rot=0)
axes[1,2].set_title('Difficulty Walking vs Disease')
axes[1,2].legend(['No Disease','Heart Disease'])

plt.suptitle('Key Risk Factors by Heart Disease Status — 2,059 CDC Patients',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 2.6 Correlation Heatmap

In [ ]:
df_temp = df.copy()
df_temp['HeartDisease_num'] = (df_temp['HeartDisease']=='Yes').astype(int)
numeric_cols = df_temp.select_dtypes(include=[np.number]).columns

plt.figure(figsize=(10, 8))
corr = df_temp[numeric_cols].corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 3. Preprocessing

We prepare the data for model training in three steps:
1. **Encode** — convert text columns to numbers
2. **Split** — 80% training, 20% test (stratified to preserve disease rate)
3. **Scale** — StandardScaler so all features have equal weight

### Feature Encoding

| Feature | Encoding | Reason |
|---|---|---|
| Yes/No columns (8 columns) | 0 or 1 | Binary |
| Sex | 0 = Female, 1 = Male | Binary |
| AgeCategory | 0 (18–24) → 12 (80+) | Ordinal — natural age order |
| GenHealth | 0 (Poor) → 4 (Excellent) | Ordinal — natural health order |
| Diabetic | 0 = No, 1 = Borderline, 2 = Yes | Ordinal |
| Race | One-hot encoding | Nominal — no natural order, avoid false ranking |

> **Data leakage rule:** StandardScaler is fitted on training data ONLY.
> The same scaling parameters are then applied to the test set.
> Fitting on all data would let test information influence training — this is called
> data leakage and would give falsely optimistic results.


In [ ]:
df_enc = df.copy()

# ── Binary Yes/No columns → 0 / 1 ────────────────────────────────────────────
binary_cols = ['Smoking','AlcoholDrinking','Stroke','DiffWalking',
               'PhysicalActivity','Asthma','KidneyDisease','SkinCancer']
for col in binary_cols:
    df_enc[col] = (df_enc[col] == 'Yes').astype(int)

df_enc['Sex'] = (df_enc['Sex'] == 'Male').astype(int)

# ── Ordinal encoding ──────────────────────────────────────────────────────────
age_map = {'18-24':0,'25-29':1,'30-34':2,'35-39':3,'40-44':4,'45-49':5,
           '50-54':6,'55-59':7,'60-64':8,'65-69':9,'70-74':10,'75-79':11,'80 or older':12}
df_enc['AgeCategory'] = df_enc['AgeCategory'].map(age_map)

gh_map = {'Poor':0,'Fair':1,'Good':2,'Very good':3,'Excellent':4}
df_enc['GenHealth'] = df_enc['GenHealth'].map(gh_map)

diab_map = {'No':0,'No, borderline diabetes':1,'Yes (during pregnancy)':1,'Yes':2}
df_enc['Diabetic'] = df_enc['Diabetic'].map(diab_map)

# ── Race: one-hot (drop first to avoid multicollinearity) ─────────────────────
race_dummies = pd.get_dummies(df_enc['Race'], prefix='Race', drop_first=True).astype(int)
df_enc = pd.concat([df_enc.drop('Race', axis=1), race_dummies], axis=1)

# ── Target: Yes → 1, No → 0 ──────────────────────────────────────────────────
df_enc['HeartDisease'] = (df_enc['HeartDisease'] == 'Yes').astype(int)

print(f"Encoded shape:  {df_enc.shape}")
print(f"Features:       {df_enc.shape[1]-1}")
print(f"Missing values: {df_enc.isnull().sum().sum()}")
df_enc.head()


In [ ]:
# ── Split ─────────────────────────────────────────────────────────────────────
X = df_enc.drop('HeartDisease', axis=1)
y = df_enc['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # preserves 8.6% disease rate in both subsets
)

print(f"Training:  {len(X_train)} patients ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test:      {len(X_test)} patients ({len(X_test)/len(X)*100:.0f}%)")
print(f"Train disease rate: {y_train.mean()*100:.1f}%")
print(f"Test disease rate:  {y_test.mean()*100:.1f}%")

# ── Scale ─────────────────────────────────────────────────────────────────────
scaler = StandardScaler()

# fit_transform: learns mean & std FROM training data, then scales it
X_train_scaled = scaler.fit_transform(X_train)

# transform only: applies the SAME parameters learned from training — no leakage
X_test_scaled  = scaler.transform(X_test)

print("\nScaling complete.")


---
## 4. Model Training with GridSearchCV

### Three Classification Algorithms

| Model | How it works | Why included |
|---|---|---|
| **Logistic Regression** | Draws a straight decision boundary. Calculates the probability of disease for each patient using a weighted sum of features. | Interpretable baseline. Good for approximately linear relationships. |
| **Random Forest** | Builds 100–300 decision trees on random subsets of the data and takes a majority vote. | Handles non-linear patterns. Provides feature importance scores. Robust to outliers. |
| **SVM (RBF kernel)** | Finds the maximum-margin boundary between classes in a transformed feature space. | Strong on smaller datasets. Effective when features are correlated. |

### GridSearchCV
GridSearchCV systematically tests every combination of settings from a predefined grid,
evaluating each using 5-fold cross-validation. It returns the combination that achieves
the best F1-score — the right metric for imbalanced medical data.

### class_weight='balanced'
All models use `class_weight='balanced'`. This tells the algorithm to penalise
mistakes on disease patients more heavily — proportional to how rare they are (8.6%).
Without this, all three models would learn to predict "no disease" almost every time
to exploit the class imbalance.

### 5-Fold Stratified Cross-Validation
The training set is divided into 5 equal parts. Each part takes a turn as the
validation set while the other 4 train the model. Results are averaged.
Stratified means each fold preserves the 8.6% disease rate.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Logistic Regression ───────────────────────────────────────────────────────
# C: smaller = stronger regularisation (simpler model)
#    larger  = weaker regularisation (fits training data more closely)
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    param_grid={'C': [0.01, 0.1, 0.5, 1, 5, 10]},
    cv=cv, scoring='f1', n_jobs=-1
)
lr_grid.fit(X_train_scaled, y_train)
lr_best = lr_grid.best_estimator_
print(f"Logistic Regression  best C:      {lr_grid.best_params_['C']}")

# ── Random Forest ─────────────────────────────────────────────────────────────
# n_estimators:      number of trees
# max_depth:         maximum depth per tree (None = unlimited, but risks overfitting)
# min_samples_split: minimum patients needed at a node before splitting
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid={
        'n_estimators':      [100, 200, 300],
        'max_depth':         [5, 10, 20, None],
        'min_samples_split': [2, 5, 10]
    },
    cv=cv, scoring='f1', n_jobs=-1
)
rf_grid.fit(X_train_scaled, y_train)
rf_best = rf_grid.best_estimator_
print(f"Random Forest        best params: {rf_grid.best_params_}")

# ── SVM ───────────────────────────────────────────────────────────────────────
# C:     margin vs misclassification trade-off
# gamma: reach of each training point's influence
# probability=True: required to generate probability scores for ROC curves
svm_grid = GridSearchCV(
    SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'),
    param_grid={
        'C':     [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.01, 0.001]
    },
    cv=cv, scoring='f1', n_jobs=-1
)
svm_grid.fit(X_train_scaled, y_train)
svm_best = svm_grid.best_estimator_
print(f"SVM                  best params: {svm_grid.best_params_}")


### Evaluate on the Held-Out Test Set

In [ ]:
models_tuned = {
    'Logistic Regression':    lr_best,
    'Random Forest':          rf_best,
    'Support Vector Machine': svm_best
}

results = {}

print(f"{'Model':<30} {'CV Acc':>8} {'±Std':>6} {'Test Acc':>10} {'Recall':>8} {'AUC':>7}")
print('─' * 75)

for name, model in models_tuned.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    y_pred    = model.predict(X_test_scaled)
    y_prob    = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc   = auc(fpr, tpr)
    rec       = recall_score(y_test, y_pred)

    results[name] = {
        'cv_mean':  cv_scores.mean(),
        'cv_std':   cv_scores.std(),
        'test_acc': accuracy_score(y_test, y_pred),
        'recall':   rec,
        'f1':       f1_score(y_test, y_pred),
        'auc':      roc_auc,
        'y_pred':   y_pred,
        'y_prob':   y_prob,
        'model':    model
    }

    print(f"{name:<30} {cv_scores.mean():>8.4f} {cv_scores.std():>6.4f} "
          f"{accuracy_score(y_test,y_pred):>10.4f} {rec:>8.4f} {roc_auc:>7.4f}")

print('\nEvaluation complete.')


In [ ]:
# Detailed classification report for each model
for name, info in results.items():
    print(f"\n{'='*55}")
    print(f" {name}")
    print(f"{'='*55}")
    print(classification_report(
        y_test, info['y_pred'],
        target_names=['No Disease', 'Heart Disease'],
        digits=3
    ))


---
## 5. Result Visualisations

### 5.1 Confusion Matrices

Every prediction falls into one of four categories:

| | Predicted No Disease | Predicted Disease |
|---|---|---|
| **Actual No Disease** | ✅ True Negative | ⚠️ False Positive |
| **Actual Disease** | 🚨 False Negative ← most dangerous | ✅ True Positive |

A **false negative** means we told a sick patient they are healthy.
This is the most dangerous error in a medical context — we want this as low as possible.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, info) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, info['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Heart Disease']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f'{name}\nAcc: {info["test_acc"]:.1%}  Recall: {info["recall"]:.1%}  FN: {fn}')

plt.suptitle(
    f'Confusion Matrices — All Models (n={len(y_test)} test patients)',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()


### 5.2 ROC Curves

The ROC curve plots how well the model separates sick from healthy patients
at every possible classification threshold.

- **AUC = 0.5** → random guessing
- **AUC > 0.8** → good discrimination
- **AUC = 1.0** → perfect


In [ ]:
plt.figure(figsize=(8, 6))
colors_roc = ['#3498db', '#e74c3c', '#2ecc71']

for (name, info), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, info['y_prob'])
    plt.plot(fpr, tpr, color=color, lw=2.5,
             label=f'{name}  (AUC = {info["auc"]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Baseline  (AUC = 0.500)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves — 2,059 CDC Patients (class_weight=balanced)', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 5.3 Model Comparison Chart

In [ ]:
model_names  = list(results.keys())
short_names  = ['Logistic Reg.', 'Random Forest', 'SVM']
cv_means     = [results[m]['cv_mean']  for m in model_names]
test_accs    = [results[m]['test_acc'] for m in model_names]
recalls      = [results[m]['recall']   for m in model_names]
aucs         = [results[m]['auc']      for m in model_names]

x     = np.arange(len(model_names))
width = 0.22

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy comparison
b1 = axes[0].bar(x-width/2, cv_means,  width, label='CV Accuracy',   color='#3498db', edgecolor='black', alpha=0.85)
b2 = axes[0].bar(x+width/2, test_accs, width, label='Test Accuracy',  color='#e74c3c', edgecolor='black', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(short_names)
axes[0].set_ylim([0.6, 0.95]); axes[0].set_ylabel('Accuracy')
axes[0].set_title('CV vs Test Accuracy', fontweight='bold')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
for b in list(b1)+list(b2):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                 f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# Recall + AUC
b3 = axes[1].bar(x-width/2, recalls, width, label='Disease Recall', color='#b91c1c', edgecolor='black', alpha=0.85)
b4 = axes[1].bar(x+width/2, aucs,    width, label='AUC-ROC',        color='#15803d', edgecolor='black', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(short_names)
axes[1].set_ylim([0.4, 1.0]); axes[1].set_ylabel('Score')
axes[1].set_title('Disease Recall & AUC-ROC', fontweight='bold')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
for b in list(b3)+list(b4):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.008,
                 f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


### 5.4 Feature Importance (Random Forest)

Which features drive the predictions most?
Longer bar = more influential across all trees.
**Red = top 5 most important.**


In [ ]:
fi = pd.DataFrame({
    'Feature':    X.columns,
    'Importance': rf_best.feature_importances_
}).sort_values('Importance', ascending=True)

bar_colors = ['#e74c3c' if i >= len(fi) - 5 else '#3498db'
              for i in range(len(fi))]

plt.figure(figsize=(9, 7))
plt.barh(fi['Feature'], fi['Importance'], color=bar_colors, edgecolor='black')
plt.xlabel('Importance Score (Gini Mean Decrease in Impurity)')
plt.title('Random Forest — Feature Importance\n'
          'Red = top 5 most influential features', fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('Top 10 features:')
for i, (_, r) in enumerate(fi.iloc[::-1].head(10).iterrows(), 1):
    print(f'  {i:2}. {r["Feature"]:<20} {r["Importance"]:.4f} ({r["Importance"]*100:.1f}%)')


---
## 6. Final Summary

In [ ]:
print('='*75)
print('FINAL MODEL COMPARISON — 2,059 CDC PATIENTS — GridSearchCV + class_weight=balanced')
print('='*75)
print(f"{'Model':<30} {'CV Acc':>8} {'±Std':>6} {'Test Acc':>10} {'Recall':>8} {'AUC':>7}")
print('─'*75)

best_name = max(results, key=lambda m: results[m]['auc'])   # best by AUC

for name, info in results.items():
    tag = ' <-- BEST' if name == best_name else ''
    print(f"{name:<30} {info['cv_mean']:>8.4f} {info['cv_std']:>6.4f} "
          f"{info['test_acc']:>10.4f} {info['recall']:>8.4f} {info['auc']:>7.4f}{tag}")

print('='*75)
b = results[best_name]
print(f'\nBest model (by AUC): {best_name}')
print(f'Test accuracy:        {b["test_acc"]:.1%}')
print(f'Disease recall:       {b["recall"]:.1%}')
print(f'AUC-ROC:              {b["auc"]:.3f}')
print()
print('Dataset:  CDC BRFSS 2020 — 2,059 patients at real-world disease rate (8.6%)')
print('Imbalance: handled via class_weight=balanced')
print('Source:   Kamilpytlak (2021) / CDC BRFSS 2020')


---
## 7. Ethical Discussion

### 7.1 Bias & Fairness
The BRFSS includes race and demographic variables. Disparities in healthcare access,
health literacy, and reporting behaviour across racial and socioeconomic groups may
cause the model to perform differently for different populations. Any deployment must
evaluate performance separately by subgroup (Barocas et al., 2019).

### 7.2 Self-Reported Data
Unlike clinical datasets, BRFSS data comes from a telephone survey. Respondents may
under-report behaviours (smoking, alcohol), have undiagnosed conditions, or recall
inaccurately. This introduces noise — a well-understood limitation of survey-based
health research that partially explains why accuracy is ~82% rather than ~90%+.

### 7.3 Class Imbalance — Honest Reporting
The real CDC disease rate is 8.6%. We preserved this in our sample and handled it
correctly with `class_weight='balanced'`. This is more honest and clinically meaningful
than a forced 50/50 split. However, recall for the minority class is still limited by the
small number of disease cases (177 in our sample, 35 in the test set).

### 7.4 Risk of False Negatives
A false negative — predicting no disease when disease is present — could delay treatment.
This is why we report disease recall separately and chose F1 as the tuning metric.
In a real deployment, lowering the classification threshold below 0.5 would increase
recall at the cost of more false alarms — a clinically reasonable trade-off in screening.

### 7.5 Privacy & Regulatory Compliance
EU GDPR and US HIPAA apply to any system making automated health decisions.
Explicit informed consent, data minimisation, and the right to explanation are required
(European Parliament, 2016).

### 7.6 Decision Support Only
This model is a decision-support tool. The physician always makes the final call.
SHAP (Lundberg & Lee, 2017) would allow clinicians to see exactly why a patient
was flagged — essential for clinical trust and regulatory compliance.

---
### References
- Barocas, S., Hardt, M., & Narayanan, A. (2019). *Fairness and machine learning.* fairmlbook.org.
- Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32.
- CDC. (2020). *Behavioral Risk Factor Surveillance System.* https://www.cdc.gov/brfss/
- Cortes, C., & Vapnik, V. (1995). Support-vector networks. *Machine Learning, 20*(3), 273–297.
- European Parliament. (2016). *General Data Protection Regulation (GDPR).*
- Kamilpytlak. (2021). *Personal key indicators of heart disease.* Kaggle. https://www.kaggle.com/datasets/kamilpytlak/personal-key-indicators-of-heart-disease
- Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *NeurIPS, 30.*
- Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *JMLR, 12*, 2825–2830.
- World Health Organization. (2023). *Cardiovascular diseases fact sheet.* https://www.who.int
